In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# Set random seed for reproducibility
np.random.seed(42)

# Step 1: Generate original synthetic dataset (1000 rows)
def generate_original_dataset(n=1000):
    data = {
        'age': np.random.randint(18, 60, n),
        'gender': np.random.choice(['M', 'F'], n, p=[0.6, 0.4]),
        'exercise_type': np.random.choice(['Squats', 'Bench Press', 'Deadlifts'], n, p=[0.4, 0.3, 0.3]),
        'sets': np.random.randint(1, 7, n),
        'reps': np.random.randint(5, 16, n),
        'weight': np.round(np.random.uniform(20, 200, n), 1),
        'frequency': np.random.randint(1, 6, n),
        'protein': np.round(np.random.uniform(80, 200, n), 1),
        'calories': np.random.randint(1500, 4000, n),
        'sleep': np.round(np.random.uniform(5, 10, n), 1),
        'experience': np.random.choice(['Beginner', 'Intermediate', 'Advanced'], n, p=[0.3, 0.5, 0.2])
    }
    df = pd.DataFrame(data)
    
    # Calculate muscle growth (1–5 cm², adjusted for experience and other factors)
    def calculate_muscle_growth(row):
        base_growth = np.random.uniform(1.0, 5.0)
        if row['experience'] == 'Beginner':
            base_growth *= 1.5
        elif row['experience'] == 'Advanced':
            base_growth *= 0.7
        if row['exercise_type'] in ['Squats', 'Deadlifts']:
            base_growth *= 1.2
        if row['gender'] == 'M':
            base_growth *= 1.1
        if row['age'] < 25:
            base_growth *= 1.1
        elif row['age'] > 40:
            base_growth *= 0.9
        if row['protein'] < 100:
            base_growth *= 0.9
        elif row['protein'] > 160:
            base_growth *= 1.1
        if row['calories'] < 2000:
            base_growth *= 0.9
        elif row['calories'] > 3000:
            base_growth *= 1.1
        if row['sleep'] < 7:
            base_growth *= 0.9
        elif row['sleep'] > 8:
            base_growth *= 1.1
        noise = np.random.normal(0, 0.3, 1)[0]
        return round(max(1.0, base_growth + noise), 2)
    
    df['muscle_size_increase_cm2'] = df.apply(calculate_muscle_growth, axis=1)
    return df

# Step 2: Add 100 rows for newbie gains
def add_newbie_gains_rows(df, n=100):
    data = {
        'age': np.random.randint(18, 41, n),  # Younger age range for newbies
        'gender': np.random.choice(['M', 'F'], n, p=[0.6, 0.4]),
        'exercise_type': np.random.choice(['Squats', 'Bench Press', 'Deadlifts'], n, p=[0.4, 0.3, 0.3]),
        'sets': np.random.randint(2, 7, n),
        'reps': np.random.randint(6, 13, n),
        'weight': np.round(np.random.uniform(20, 150, n), 1),  # Lighter weights
        'frequency': np.random.randint(2, 5, n),
        'protein': np.round(np.random.uniform(100, 180, n), 1),
        'calories': np.random.randint(2000, 3501, n),
        'sleep': np.round(np.random.uniform(6, 9, n), 1),
        'experience': ['Beginner'] * n
    }
    newbie_df = pd.DataFrame(data)
    
    # Calculate newbie gains (4–8 cm²)
    def calculate_newbie_muscle_growth(row):
        base_growth = np.random.uniform(4.0, 8.0)
        if row['exercise_type'] in ['Squats', 'Deadlifts']:
            base_growth *= 1.2
        if row['gender'] == 'M':
            base_growth *= 1.1
        if row['age'] < 25:
            base_growth *= 1.1
        elif row['age'] > 35:
            base_growth *= 0.9
        if row['protein'] < 120:
            base_growth *= 0.9
        elif row['protein'] > 160:
            base_growth *= 1.1
        if row['calories'] < 2500:
            base_growth *= 0.9
        elif row['calories'] > 3000:
            base_growth *= 1.1
        if row['sleep'] < 7:
            base_growth *= 0.9
        elif row['sleep'] > 8:
            base_growth *= 1.1
        noise = np.random.normal(0, 0.3, 1)[0]
        return round(max(4.0, base_growth + noise), 2)
    
    newbie_df['muscle_size_increase_cm2'] = newbie_df.apply(calculate_newbie_muscle_growth, axis=1)
    return pd.concat([df, newbie_df], ignore_index=True)

# Step 3: Predict muscle growth
def predict_muscle_growth(age, gender, exercise_type, sets, reps, weight, frequency, protein, calories, sleep, experience):
    try:
        # Load and preprocess dataset
        df = generate_original_dataset()
        df = add_newbie_gains_rows(df)
        
        le_gender = LabelEncoder()
        le_exercise = LabelEncoder()
        le_experience = LabelEncoder()
        df['gender'] = le_gender.fit_transform(df['gender'])
        df['exercise_type'] = le_exercise.fit_transform(df['exercise_type'])
        df['experience'] = le_experience.fit_transform(df['experience'])
        
        features = ['age', 'gender', 'exercise_type', 'sets', 'reps', 'weight', 
                    'frequency', 'protein', 'calories', 'sleep', 'experience']
        X = df[features]
        y = df['muscle_size_increase_cm2']
        
        # Train model
        model = RandomForestRegressor(n_estimators=100, random_state=42)
        model.fit(X, y)
        
        # Validate inputs
        if not (18 <= age <= 100):
            raise ValueError("Age must be between 18 and 100.")
        if gender not in le_gender.classes_:
            raise ValueError(f"Gender must be one of {list(le_gender.classes_)}.")
        if exercise_type not in le_exercise.classes_:
            raise ValueError(f"Exercise type must be one of {list(le_exercise.classes_)}.")
        if experience not in le_experience.classes_:
            raise ValueError(f"Experience level must be one of {list(le_experience.classes_)}.")
        if not (1 <= sets <= 10):
            raise ValueError("Sets must be between 1 and 10.")
        if not (1 <= reps <= 20):
            raise ValueError("Reps must be between 1 and 20.")
        if not (0 < weight <= 300):
            raise ValueError("Weight must be between 0 and 300 kg.")
        if not (1 <= frequency <= 7):
            raise ValueError("Frequency must be between 1 and 7 days per week.")
        if not (0 < protein <= 300):
            raise ValueError("Protein intake must be between 0 and 300 grams.")
        if not (0 < calories <= 5000):
            raise ValueError("Calories must be between 0 and 5000 kcal.")
        if not (0 < sleep <= 24):
            raise ValueError("Sleep must be between 0 and 24 hours.")
        
        # Encode categorical inputs
        gender_encoded = le_gender.transform([gender])[0]
        exercise_encoded = le_exercise.transform([exercise_type])[0]
        experience_encoded = le_experience.transform([experience])[0]
        
        # Create input array
        input_data = np.array([[age, gender_encoded, exercise_encoded, sets, reps, weight, 
                               frequency, protein, calories, sleep, experience_encoded]])
        
        # Predict
        prediction = model.predict(input_data)[0]
        return round(prediction, 2)
    except Exception as e:
        return f"Error: {str(e)}"

# Generate and save dataset
df = generate_original_dataset()
df = add_newbie_gains_rows(df)
df.to_csv('synthesized_muscle_growth_with_newbie_gains.csv', index=False)

# Print dataset summary
print(f"Dataset size: {len(df)} rows")
print("\nSample of newbie gains rows:")
print(df[df['experience'] == 'Beginner'].tail())



Dataset size: 1100 rows

Sample of newbie gains rows:
      age gender exercise_type  sets  reps  weight  frequency  protein  \
1095   20      M     Deadlifts     2     6   125.5          2    119.9   
1096   22      F        Squats     3     8    34.3          3    107.3   
1097   34      M   Bench Press     6    11    35.8          3    160.0   
1098   25      M        Squats     6     9    97.3          2    114.0   
1099   34      F        Squats     2     6    64.8          3    129.8   

      calories  sleep experience  muscle_size_increase_cm2  
1095      2556    8.0   Beginner                      7.43  
1096      2113    7.0   Beginner                      7.67  
1097      2854    7.7   Beginner                      6.01  
1098      2675    6.9   Beginner                      5.76  
1099      2976    7.2   Beginner                      7.44  


In [3]:
# Example prediction
result = predict_muscle_growth(
    age=50, gender='M', exercise_type='Squats', sets=4, reps=10, weight=80,
    frequency=3, protein=140, calories=2800, sleep=8, experience='Intermediate'
)
print(f"\nPredicted muscle growth after 12 weeks: {result} cm²")


Predicted muscle growth after 12 weeks: 4.31 cm²


c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [4]:
import torch
import torch.onnx

ModuleNotFoundError: No module named 'torch'

In [5]:
%pip install torch onnx

  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/44/80/b353c024e6b624cd9ce1d66dcb9d24e0294680f95b369f19280e241a0159/torch-2.7.0-cp312-cp312-win_amd64.whl.metadata
  Using cached torch-2.7.0-cp312-cp312-win_amd64.whl.metadata (29 kB)
  Obtaining dependency information for onnx from https://files.pythonhosted.org/packages/e8/92/048ba8fafe6b2b9a268ec2fb80def7e66c0b32ab2cae74de886981f05a27/onnx-1.18.0-cp312-cp312-win_amd64.whl.metadata
  Using cached onnx-1.18.0-cp312-cp312-win_amd64.whl.metadata (7.0 kB)
  Obtaining dependency information for networkx from https://files.pythonhosted.org/packages/b9/54/dd730b32ea14ea797530a4479b2ed46a6fb250f682a9cfb997e968bf0261/networkx-3.4.2-py3-none-any.whl.metadata
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
Using cached torch-2.7.0-cp312-cp312-win_amd64.whl (212.5 MB)
Using cached onnx-1.18.0-cp312-cp312-win_amd64.whl (15.9 MB)
Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
Note: you

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'c:\\Python312\\Scripts\\backend-test-tools.exe' -> 'c:\\Python312\\Scripts\\backend-test-tools.exe.deleteme'


[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
